# 08 — Statistical Analysis

This notebook runs the core statistical tests that characterise script-level bias in neural MT evaluation metrics.

All tests operate directly on the per-segment data produced by earlier notebooks in this pipeline. The notebook is designed to be self-contained: it loads data, runs every test, prints results, and writes a CSV summary — no cached values are used.

**Tests performed**

| # | Question | Method |
|---|----------|--------|
| 1–4 | Does script family explain COMET variance, and does romanisation collapse that variance? | One-way ANOVA + η² |
| 5–6 | How much do language-level mean COMET scores shift under romanisation? | Mean comparison + delta |
| 7–8 | Does COMET-MQM alignment change with script? | Spearman ρ per language |
| 9 | Does romanisation affect COMET's ability to discriminate error severity? | Severity-range comparison |
| 10–11 | Are TP and IP anti-correlated across languages? | Pearson r (n = 5) |
| 12 | Does sentence-level SBI rank languages in the same order as COMET? | Spearman ρ (n = 5) |
| 13 | Is IP correlated with COMET at the sentence level? | Pearson r per language |

**Prerequisites:** `Information_parity_outputs_all.xlsx` must be present in `../../data/processed/`.

## Setup

Install dependencies if needed:

```bash
pip install pandas numpy scipy openpyxl
```

The cell below imports all required libraries and defines the paths and column-name constants used throughout the notebook. Column names are defined once here — if the upstream data file ever renames a column, only this cell needs updating.

In [ ]:
from __future__ import annotations

import io
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

REPO_ROOT   = Path("../..").resolve()
DATA_PROC   = REPO_ROOT / "data" / "processed"
DATA_TOK    = DATA_PROC / "tokenization_outputs"
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

LANGUAGES = ["gujarati", "hindi", "malayalam", "marathi", "tamil"]

# Script-family grouping used in the ANOVA
SCRIPT_GROUP = {
    "hindi":     "Devanagari",
    "marathi":   "Devanagari",
    "gujarati":  "non-Devanagari",
    "tamil":     "non-Devanagari",
    "malayalam": "non-Devanagari",
}

# ── Column name constants (actual xlsx column names) ──────────────────────
COL_COMET_NAT = "COMET"           # native-script COMET
COL_COMET_ROM = "COMET_romanized" # romanised COMET

# MQM-derived human score (0-25 scale) — column name in workbook
COL_MQM = "Human_scores"

# Error severity is not stored as a single column; it must be derived from the
# five per-error columns Error1_Severity … Error5_Severity.  The derivation
# helper below creates COL_SEVERITY ("Error_Severity") on every dataframe after
# loading.  Uses the worst (most severe) non-null rating across the five slots.
COL_SEVERITY     = "Error_Severity"          # name of the derived column
ERROR_SEV_COLS   = [f"Error{i}_Severity" for i in range(1, 6)]  # source cols
SEV_RANK         = {"Very High": 0, "High": 1, "Medium": 2,
                    "Low": 3, "Very Low": 4}  # 0 = most severe

def derive_severity(df: pd.DataFrame) -> pd.DataFrame:
    """Add COL_SEVERITY column = worst severity across Error1..5_Severity.

    If none of the source columns exist the column is not added (downstream
    tests guard with `if COL_SEVERITY not in df.columns`).
    """
    present = [c for c in ERROR_SEV_COLS if c in df.columns]
    if not present:
        return df
    # Map each cell to its numeric rank; take the minimum (= most severe)
    ranked = df[present].apply(lambda col: col.map(SEV_RANK))
    worst_rank = ranked.min(axis=1)                         # NaN where all blank
    rank_to_sev = {v: k for k, v in SEV_RANK.items()}
    df = df.copy()
    df[COL_SEVERITY] = worst_rank.map(rank_to_sev)          # back to string label
    return df

# TP / IP columns (confirmed present in all five sheets)
COL_TP_NAT = "Translation_xlmr_TP"
COL_TP_ROM = "Translation_Transliteration_romanized_xlmr_TP"
COL_IP_NAT = "Translation_xlmr_IP"
COL_IP_ROM = "Translation_Transliteration_romanized_xlmr_IP"

# Severity ordering (worst to best) — used in Test 9
SEV_ORDER = ["Very High", "High", "Medium", "Low", "Very Low"]

print("Configuration loaded.")
print(f"  Repo root : {REPO_ROOT}")
print(f"  Data dir  : {DATA_PROC}")
print(f"  Results   : {RESULTS_DIR}")

## Loading Data

Loads all five language sheets directly from `Information_parity_outputs_all.xlsx` — the single source of truth. Each sheet contains metric scores, XLM-R token counts, TP/IP columns, and human scores. The Malayalam `Chrf` column is normalised to `chrF` and any stray `Unnamed:` columns are dropped.

In [ ]:
# Load directly from the workbook — single source of truth
XLSX = DATA_PROC / "Information_parity_outputs_all.xlsx"
if not XLSX.exists():
    XLSX = DATA_TOK / "Information_parity_outputs_all.xlsx"
if not XLSX.exists():
    raise FileNotFoundError(
        f"Information_parity_outputs_all.xlsx not found in {DATA_PROC} or {DATA_TOK}."
    )

SHEET_MAP = {
    'gujarati':  'Indic_mt _for_analysis - Gujara',
    'hindi':     'Indic_mt _for_analysis - Hindi_',
    'malayalam': 'Indic_mt _for_analysis - Malaya',
    'marathi':   'Indic_mt _for_analysis - Marath',
    'tamil':     'Indic_mt _for_analysis - Tamil_',
}

data: dict[str, pd.DataFrame] = {}
for lang in LANGUAGES:
    df = pd.read_excel(XLSX, sheet_name=SHEET_MAP[lang])
    if 'Chrf' in df.columns and 'chrF' not in df.columns:
        df = df.rename(columns={'Chrf': 'chrF'})
    df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
    data[lang] = df

data = {lang: derive_severity(df) for lang, df in data.items()}

print(f"Loaded {len(data)} language sheets from {XLSX.name}.")
for lang, df in data.items():
    sev_ok = "✓" if COL_SEVERITY in df.columns else "✗"
    print(f"  {lang:<12} {len(df):>5} rows  |  severity={sev_ok}")

## One-Way ANOVA Helper

The `one_way_anova_eta2` function computes a one-way ANOVA and returns the F-statistic, p-value, and effect size η² (eta-squared).

η² is defined as the ratio of between-group variance to total variance:

$$\eta^2 = \frac{SS_{\text{between}}}{SS_{\text{total}}}$$

An η² close to 0 means group membership (script family) explains almost none of the variance in COMET scores. An η² close to 1 means it explains nearly all of it.

In [ ]:
def one_way_anova_eta2(groups: dict[str, pd.Series]) -> dict:
    arrays     = [np.asarray(v, dtype=float) for v in groups.values()]
    k          = len(arrays)
    N          = sum(len(a) for a in arrays)
    grand_mean = np.concatenate(arrays).mean()

    ss_between = sum(len(a) * (a.mean() - grand_mean) ** 2 for a in arrays)
    ss_within  = sum(((a - a.mean()) ** 2).sum() for a in arrays)
    ss_total   = ss_between + ss_within

    df_between = k - 1
    df_within  = N - k

    F    = (ss_between / df_between) / (ss_within / df_within)
    p    = stats.f.sf(F, df_between, df_within)
    eta2 = ss_between / ss_total

    return dict(F=F, p=p, eta2=eta2, df_between=df_between, df_within=df_within)


def fmt_p(p: float) -> str:
    if p < 1e-99:
        return "< 10\u207b\u2079\u2079"
    if p < 0.001:
        return f"= {p:.2e}"
    return f"= {p:.4f}"


print("ANOVA helper defined.")

## Tests 1–4 — Script-Family ANOVA on COMET

We group the five languages into two script families (Devanagari: Hindi, Marathi; non-Devanagari: Gujarati, Tamil, Malayalam) and run a one-way ANOVA on COMET scores under both the native-script and romanised conditions.

The key question: does script family account for a meaningful share of COMET variance, and does that share collapse when all text is romanised? η² quantifies how much of the total COMET score variance is explained purely by which script family the language belongs to.

In [ ]:
nat_by_script: dict[str, list] = {"Devanagari": [], "non-Devanagari": []}
rom_by_script: dict[str, list] = {"Devanagari": [], "non-Devanagari": []}

for lang, df in data.items():
    grp = SCRIPT_GROUP[lang]
    nat_by_script[grp].extend(df[COL_COMET_NAT].dropna().tolist())
    rom_by_script[grp].extend(df[COL_COMET_ROM].dropna().tolist())

anova_nat = one_way_anova_eta2(nat_by_script)
anova_rom = one_way_anova_eta2(rom_by_script)

eta2_nat_pct  = anova_nat["eta2"] * 100
eta2_rom_pct  = anova_rom["eta2"] * 100
reduction_pct = (1 - anova_rom["eta2"] / anova_nat["eta2"]) * 100

print("One-way ANOVA: script family vs COMET score")
print()
print(f"  Native-script  F = {anova_nat['F']:>9,.1f}  p {fmt_p(anova_nat['p'])}  \u03b7\u00b2 = {eta2_nat_pct:.1f}%")
print(f"  Romanised      F = {anova_rom['F']:>9,.1f}  p {fmt_p(anova_rom['p'])}  \u03b7\u00b2 = {eta2_rom_pct:.1f}%")
print(f"  \u03b7\u00b2 reduction   = {reduction_pct:.1f}%")

## Tests 5–6 — Language-Level Mean COMET Shift

For each language we compute the mean COMET score under native script and under romanisation, then take the difference (delta). A positive delta means COMET increased after romanisation; a negative delta means it fell.

We also compute the Hindi–Gujarati gap under both conditions to quantify how much of the between-language spread is driven by script rather than translation quality.

In [ ]:
rows = []
for lang in ["gujarati", "tamil", "malayalam", "marathi", "hindi"]:
    df       = data[lang]
    mean_nat = df[COL_COMET_NAT].mean()
    mean_rom = df[COL_COMET_ROM].mean()
    delta    = mean_rom - mean_nat
    rows.append({"language": lang.capitalize(),
                 "COMET_nat": mean_nat,
                 "COMET_rom": mean_rom,
                 "delta_pts": delta})

means_df = pd.DataFrame(rows)

print("Language-level mean COMET (native vs romanised):")
print(means_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print()

guj_nat  = means_df.loc[means_df.language == "Gujarati", "COMET_nat"].item()
hin_nat  = means_df.loc[means_df.language == "Hindi",    "COMET_nat"].item()
guj_rom  = means_df.loc[means_df.language == "Gujarati", "COMET_rom"].item()
hin_rom  = means_df.loc[means_df.language == "Hindi",    "COMET_rom"].item()
gap_nat  = hin_nat - guj_nat
gap_rom  = hin_rom - guj_rom

print(f"Hindi\u2013Gujarati gap (native)    : {gap_nat:+.3f}")
print(f"Hindi\u2013Gujarati gap (romanised) : {gap_rom:+.3f}")

## Tests 7–8 — Spearman ρ Between COMET and MQM

Spearman's ρ measures the rank-order correlation between COMET scores and human MQM judgements at the sentence level. We compute ρ separately for native-script and romanised COMET to test whether the metric aligns better or worse with human judgement when the script changes.

In [ ]:
spearman_rows = []
for lang in LANGUAGES:
    df = data[lang]
    if COL_MQM not in df.columns:
        print(f"  '{COL_MQM}' not found for {lang} — skipped")
        continue
    rho_nat, p_nat = stats.spearmanr(df[COL_COMET_NAT].dropna(), df[COL_MQM].dropna())
    rho_rom, p_rom = stats.spearmanr(df[COL_COMET_ROM].dropna(), df[COL_MQM].dropna())
    spearman_rows.append({
        "language": lang.capitalize(),
        "rho_nat": rho_nat, "p_nat": p_nat,
        "rho_rom": rho_rom, "p_rom": p_rom,
        "delta_rho": rho_rom - rho_nat,
    })

if spearman_rows:
    spearman_df = pd.DataFrame(spearman_rows)
    print("Spearman \u03c1 \u2014 COMET vs MQM:")
    print(spearman_df[["language", "rho_nat", "rho_rom", "delta_rho"]].to_string(
        index=False, float_format=lambda x: f"{x:.3f}"))
else:
    spearman_df = pd.DataFrame()
    print("MQM column not available \u2014 skipping Tests 7\u20138.")

## Test 9 — Severity-Discrimination Range

A useful evaluation metric should assign clearly different scores to segments with different error severities. We measure this as the difference in mean COMET score between the most severe and least severe error category for each language.

A larger range indicates better discrimination. If the range shrinks substantially after romanisation, it suggests the metric is less sensitive to translation quality in the romanised condition.

In [ ]:
def severity_range(df: pd.DataFrame, comet_col: str, lang: str) -> float:
    if COL_SEVERITY not in df.columns:
        return float("nan")
    present = (
        df.dropna(subset=[COL_SEVERITY, comet_col])
          .groupby(COL_SEVERITY)[comet_col]
          .mean()
    )
    ordered = [s for s in SEV_ORDER if s in present.index]
    if lang == "hindi":
        ordered = [s for s in SEV_ORDER if s in present.index and s not in ("Very Low", "Low")]
    if len(ordered) < 2:
        return float("nan")
    return present[ordered[0]] - present[ordered[-1]]


sev_rows = []
for lang, df in data.items():
    r_nat = severity_range(df, COL_COMET_NAT, lang)
    r_rom = severity_range(df, COL_COMET_ROM, lang)
    sev_rows.append({"language": lang.capitalize(), "range_nat": r_nat, "range_rom": r_rom,
                     "delta": r_rom - r_nat})

sev_df = pd.DataFrame(sev_rows)
print("Severity-discrimination range (highest \u2212 lowest severity):")
print(sev_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

## Tests 10–11 — TP–IP Anti-Correlation Across Languages

Tokenization Parity (TP) and Information Parity (IP) are designed to capture complementary dimensions of script bias. TP measures surface-level tokenizer fragmentation; IP measures how much harder the language model finds the text to compress. If the two metrics are genuinely anti-correlated, that provides evidence that romanisation simultaneously inflates TP (more tokens per word in Latin script) while collapsing IP (the model finds Latin-script text easier to predict).

With only five languages the Pearson r is computed on aggregated language means, so the sample size is n = 5.

In [ ]:
lang_means = {
    lang: {
        "tp_nat": df[COL_TP_NAT].mean(),
        "tp_rom": df[COL_TP_ROM].mean(),
        "ip_nat": df[COL_IP_NAT].mean(),
        "ip_rom": df[COL_IP_ROM].mean(),
    }
    for lang, df in data.items()
    if all(c in df.columns for c in [COL_TP_NAT, COL_TP_ROM, COL_IP_NAT, COL_IP_ROM])
}

if lang_means:
    lm = pd.DataFrame(lang_means).T
    r_nat, p_nat = stats.pearsonr(lm["tp_nat"], lm["ip_nat"])
    r_rom, p_rom = stats.pearsonr(lm["tp_rom"], lm["ip_rom"])
    print("Pearson r between TP and IP across languages (n = 5):")
    print(f"  Native-script  r = {r_nat:.3f}  p {fmt_p(p_nat)}")
    print(f"  Romanised      r = {r_rom:.3f}  p {fmt_p(p_rom)}")
    print()
    print("Language-level means:")
    print(lm.round(3).to_string())
else:
    lm = pd.DataFrame()
    print("TP/IP columns not found \u2014 skipping Tests 10\u201311.")

## Test 12 — SBI vs COMET Rank Order

Script Bias Index (SBI) at the sentence level is defined as the ratio of TP to IP for each segment. We average this ratio per language to get a language-level SBI score, then check whether the language ranking by SBI matches the ranking by mean COMET score.

Perfect rank correlation (ρ = 1.0) would mean that languages with higher tokenizer bias also receive systematically higher COMET scores, which would be direct evidence that TP/IP-level script bias propagates into the final metric output.

In [ ]:
sbi_rows = []
for lang, df in data.items():
    if not all(c in df.columns for c in [COL_TP_NAT, COL_IP_NAT, COL_COMET_NAT]):
        continue
    sbi        = (df[COL_TP_NAT] / df[COL_IP_NAT]).mean()
    mean_comet = df[COL_COMET_NAT].mean()
    sbi_rows.append({"language": lang.capitalize(), "SBI_nat": sbi, "COMET_nat": mean_comet})

if sbi_rows:
    sbi_df = pd.DataFrame(sbi_rows).sort_values("SBI_nat", ascending=False)
    rho_sbi, p_sbi = stats.spearmanr(sbi_df["SBI_nat"], sbi_df["COMET_nat"])
    print("SBI vs COMET rank order (n = 5):")
    print(sbi_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
    print()
    print(f"Spearman \u03c1 (SBI_nat vs COMET_nat) = {rho_sbi:.3f}  p {fmt_p(p_sbi)}")
else:
    sbi_df = pd.DataFrame()
    print("Required columns not found \u2014 skipping Test 12.")

## Test 13 — Sentence-Level IP–COMET Correlation

At the sentence level, we compute the Pearson correlation between the IP score and the COMET score for each language separately. A negative r would suggest that segments where the model finds the Indic text harder to compress (lower IP) tend to receive lower COMET scores — consistent with the hypothesis that representational difficulty propagates into metric bias.

In [ ]:
ip_comet_rows = []
for lang, df in data.items():
    if COL_IP_NAT not in df.columns or COL_COMET_NAT not in df.columns:
        continue
    mask = df[[COL_IP_NAT, COL_COMET_NAT]].notna().all(axis=1)
    sub  = df[mask]
    if len(sub) < 2:
        continue
    r, p = stats.pearsonr(sub[COL_IP_NAT], sub[COL_COMET_NAT])
    ip_comet_rows.append({"language": lang.capitalize(), "r": r, "p": p,
                          "n": len(sub), "sig": "\u2713" if p < 0.05 else ""})

if ip_comet_rows:
    ip_comet_df = pd.DataFrame(ip_comet_rows)
    print("Pearson r: IP_nat vs COMET_nat (sentence level)")
    print(ip_comet_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
else:
    ip_comet_df = pd.DataFrame()
    print("Required columns not available.")

## Summary Output

All computed results are assembled into a single CSV file and saved to `../../results/`. This file consolidates the ANOVA results, language-level means, Spearman correlations, and SBI–COMET rank comparison in one place for downstream use.

In [ ]:
summary_rows = []

for lang in LANGUAGES:
    row = {"language": lang.capitalize()}
    df  = data[lang]

    row["COMET_nat"] = df[COL_COMET_NAT].mean()
    row["COMET_rom"] = df[COL_COMET_ROM].mean()
    row["delta_pts"] = row["COMET_rom"] - row["COMET_nat"]

    if not spearman_df.empty:
        s = spearman_df[spearman_df.language == lang.capitalize()]
        if not s.empty:
            row["rho_nat"] = s.iloc[0]["rho_nat"]
            row["rho_rom"] = s.iloc[0]["rho_rom"]

    for col, key in [(COL_TP_NAT, "TP_nat"), (COL_TP_ROM, "TP_rom"),
                     (COL_IP_NAT, "IP_nat"), (COL_IP_ROM, "IP_rom")]:
        if col in df.columns:
            row[key] = df[col].mean()

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
out_path   = RESULTS_DIR / "08_statistical_summary.csv"
summary_df.to_csv(out_path, index=False)
print(f"Summary saved to: {out_path}")
print()
print(summary_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

## References

- **ANOVA and effect sizes:** Field, A. (2013). *Discovering Statistics Using IBM SPSS Statistics* (4th ed.). Sage.

- **Spearman rank correlation:** Spearman, C. (1904). The proof and measurement of association between two things. *American Journal of Psychology*, 15(1), 72–101.

- **IndicMT Eval dataset:** Sai B., A., Dixit, T., Nagarajan, V., Kunchukuttan, A., Kumar, P., Khapra, M. M., & Dabre, R. (2023). IndicMT Eval: A Dataset to Meta-Evaluate Machine Translation Metrics for Indian Languages. *ACL 2023*. https://aclanthology.org/2023.acl-long.795

- **COMET:** Rei, R., Stewart, C., Farinha, A. C., & Lavie, A. (2020). COMET: A Neural Framework for MT Evaluation. *EMNLP 2020*. https://aclanthology.org/2020.emnlp-main.213

- **XLM-RoBERTa:** Conneau, A., et al. (2020). Unsupervised Cross-lingual Representation Learning at Scale. *ACL 2020*. https://arxiv.org/abs/1911.02116

- **BLOOM:** BigScience Workshop. (2022). BLOOM: A 176B-Parameter Open-Access Multilingual Language Model. https://arxiv.org/abs/2211.05100